# Generating the ESM-IF1 embeddings

ESM-IF1 training lays on inverse folding and thus create structural embeddings. The objetive is to combine ESM2 evolutionnary embeddings with ESM-IF1 structural embeddings to increase the model comprehension of the protein context.

In [1]:
%pip install numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.


In [7]:
import numpy as np
print(np.__version__)

1.26.4


In [8]:
import torch
import os
import sqlite3
import numpy as np
import esm
from esm.inverse_folding.util import load_structure, extract_coords_from_structure
from esm.inverse_folding.util import CoordBatchConverter

The IEDB ID is more relevant than the PDB ID, this allows the Embeddings to be named after the IEDB ID

In [9]:
def get_iedb_from_pdb(pdb_id):
    """
    Given a PDB ID, retrieves the corresponding IEDB ID from the database.
    Args:
        pdb_id (str): The PDB identifier.
    Returns:
        str: The corresponding IEDB ID.
    """
    conn = sqlite3.connect('data/db_epitopes.db')
    cur = conn.cursor()
    cur.execute("SELECT Iedb FROM epitopes WHERE Protein LIKE ?", (f"%{pdb_id}%",))
    result = cur.fetchone()
    conn.close()
    if result:
        return result[0]
    else:
        raise ValueError(f"No IEDB ID found for PDB ID: {pdb_id}")


Loads the model and its alphabet

In [10]:
def load_ESMIF1_model(model_path):
    """
    Load the ESM-IF1 model and alphabet from a local .pt file.

    Args:
        model_path (str): Path to the local .pt file.
    Returns:
        model: The loaded ESM-IF1 model.
        alphabet: A dict of the correspondance of AA tokens to numbers. Without it, the model doesn't understand sequences.
    """
    if not os.path.exists(model_path):
        print(f"Error : File not found at : {os.path.abspath(model_path)}")
    else:
        model, alphabet = esm.pretrained.load_model_and_alphabet(model_path)
        print("Model loaded successfully!")
    return model, alphabet

Code to compute a PDB into the ESM-F1 model and save the .pt embeddings file.

In [11]:
def save_ESMIF1_embeddings(model, alphabet, pdb_path):
    """
    Generate and save ESM-IF1 embeddings for a given protein structure in PDB format.
    
    Args:
        model: The pre-trained ESM-IF1 model.
        alphabet: The corresponding alphabet for the model.
        pdb_path (str): Path to the PDB file of the protein structure: [pdb_quality]/[pdb_id]/[name_file].pdb
    Returns:
        dict: A dictionary containing the PDB ID, embeddings tensor, sequence, and dimensions.
    """
    structure = load_structure(pdb_path, "A")
    # Extract the coordinates (N, CA, C, O)
    coords, seq = extract_coords_from_structure(structure)

    # Prepare the batch (mandatory)
    batch_converter = CoordBatchConverter(alphabet)
    batch_coords, confidence, strs, tokens, padding_mask = batch_converter([(coords, None, seq)])

    # Transfer the model to the GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    batch_coords = batch_coords.to(device)
    confidence = confidence.to(device)
    padding_mask = padding_mask.to(device)

    # Give the coords to the model
    with torch.no_grad():
        encoder_out = model.encoder(batch_coords, padding_mask, confidence)
        embeddings = encoder_out['encoder_out']
        if isinstance(embeddings, list):
            embeddings = embeddings[0]
        

    # From shape (1, L+2, 51) to (L, 512)
    embeddings_final = embeddings[1:-1].squeeze(1)
    pdb_id = pdb_path.split("\\")[-2]  # Get folder name (protein ID)
    protein_id = extract_protein_id(pdb_id)
    
    print(f"Embeddings shape for {pdb_path} Protein ID={protein_id}: {embeddings_final.shape}")

    # Get quality from path
    pdb_quality = pdb_path.split(os.sep)[-3]
    
    data_to_save = {
        "pdb_id": protein_id,
        "pdb_quality": pdb_quality,
        "embeddings": embeddings_final,
        "sequence": seq,
        "dims": embeddings_final.shape
    }
    
    # Create directory if it doesn't exist
    output_dir = f"data/embeddings/ESMIF1/"
    os.makedirs(output_dir, exist_ok=True)
    
    torch.save(data_to_save, f"{output_dir}{protein_id}.pt")
    return data_to_save

Generate the embeddins for every pdb quality

In [12]:
def extract_protein_id(protein_str):
    """
    Extracts the protein ID from a full URL or reference.
    Converts: http://www.ncbi.nlm.nih.gov/protein/ADA79546.1 → ADA79546.1
    """
    if '/' in protein_str:
        return protein_str.rstrip('/').split('/')[-1]
    return protein_str

def load_proteins_from_db(db_path="data/db_epitopes.db", table="Proteins"):
    """
    Loads all protein IDs from the specified database table.
    
    Args:
        db_path (str): Path to SQLite database file.
        table (str): Name of table containing 'Protein' column.
    
    Returns:
        set: Set of cleaned protein IDs.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    query = f'SELECT Protein FROM {table}'
    rows = cursor.execute(query).fetchall()
    conn.close()
    
    # Extract and clean protein IDs
    protein_ids = {extract_protein_id(row[0]) for row in rows}
    print(f"Loaded {len(protein_ids)} proteins from {table} table")
    return protein_ids

In [ ]:
from src.utils.dataset_utils import get_rank001_pdb

# Load protein IDs from database
proteins_to_process = load_proteins_from_db("data/db_epitopes.db", "Proteins")

total = 0
processed = 0
skipped = 0
model, alphabet = load_ESMIF1_model("esm_if1_gvp4_t16_142M_UR50.pt")
save_embeddings_path = "data/embeddings/ESMIF1/"
pdb_quality = ["pdb_very_high", "pdb_high", "pdb_low", "pdb_very_low"]
pdb_path = "data/outputs_pdb/"

for quality in pdb_quality:
    path = ""
    print(f"\npdb quality: {quality}")
    path = os.path.join(pdb_path, quality)
    print(f"Processing quality: {quality} and so path: {path}")

    # In each protein folder, get the rank_001 pdb
    for folder in os.listdir(path):
        # Extract clean protein ID from folder name
        protein_id = extract_protein_id(folder)
        
        # Check if this protein is in the database
        if protein_id not in proteins_to_process:
            skipped += 1
            continue
        
        print(f"number of files in folder {folder} : {len(os.listdir(os.path.join(path, folder)))}")
        folder_path = os.path.join(path, folder)
        print(f"Processing folder: {folder_path}")
        pdb_path2 = get_rank001_pdb(folder_path)
        if pdb_path2:
            total += 1
        if pdb_path2:
            # Save the embeddings
            data_to_save = save_ESMIF1_embeddings(model, alphabet, pdb_path2)
            print(f"Embeddings saved for {folder_path} at {save_embeddings_path}/{quality}/")
            processed += 1
        else:
            print(f"Skipping {folder_path}: No 'rank_001' PDB file found.")

print(f"\n✓ Processed: {processed}")
print(f"✗ Skipped (not in database): {skipped}")
print(f"Total rank_001 files found: {total}")

Loaded 1178 proteins from Proteins table


c:\Users\BISITE\Desktop\GAN-epitope-prediction\venv\Lib\site-packages\esm\pretrained.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(str(model_l

Model loaded successfully!

pdb quality: pdb_very_high
Processing quality: pdb_very_high and so path: data/outputs_pdb/pdb_very_high
number of files in folder 1406189A : 6
Processing folder: data/outputs_pdb/pdb_very_high\1406189A
Embeddings shape for data\outputs_pdb\pdb_very_high\1406189A\1406189A_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb Protein ID=1406189A: torch.Size([250, 512])
Embeddings saved for data/outputs_pdb/pdb_very_high\1406189A at data/embeddings/ESMIF1//pdb_very_high/
number of files in folder 1503111B : 6
Processing folder: data/outputs_pdb/pdb_very_high\1503111B
Embeddings shape for data\outputs_pdb\pdb_very_high\1503111B\1503111B_unrelaxed_rank_001_alphafold2_ptm_model_3_seed_000.pdb Protein ID=1503111B: torch.Size([203, 512])
Embeddings saved for data/outputs_pdb/pdb_very_high\1503111B at data/embeddings/ESMIF1//pdb_very_high/
number of files in folder 1A2Y_C : 6
Processing folder: data/outputs_pdb/pdb_very_high\1A2Y_C
Embeddings shape for data\outputs

In [9]:
from src.utils.dataset_utils import get_rank001_pdb
total = 0
save_embeddings_path = "data/embeddings/ESMIF1/"
pdb_quality = ["pdb_very_high", "pdb_high", "pdb_low", "pdb_very_low"]
pdb_path = "data/outputs_pdb/"

for quality in pdb_quality:
    path = ""
    print(f"pdb quality: {quality}")
    path = os.path.join(pdb_path, quality)
    print(f"Processing quality: {quality} and so path: {path}")

    # In each protein folder, get the rank_001 pdb
    for folder in os.listdir(path):
        # print(f"number of files in folder {folder} : {len(os.listdir(os.path.join(path, folder)))}")
        folder_path = os.path.join(path, folder)
        folder_path = os.path.join(path, folder)
        # print(f"Processing folder: {folder_path}")
        pdb_path2 = get_rank001_pdb(folder_path)
        if pdb_path2:
            total+= 1
print(f"Total embeddings extracted: {total}")

pdb quality: pdb_very_high
Processing quality: pdb_very_high and so path: data/outputs_pdb/pdb_very_high
pdb quality: pdb_high
Processing quality: pdb_high and so path: data/outputs_pdb/pdb_high
pdb quality: pdb_low
Processing quality: pdb_low and so path: data/outputs_pdb/pdb_low
pdb quality: pdb_very_low
Processing quality: pdb_very_low and so path: data/outputs_pdb/pdb_very_low
Total embeddings extracted: 1239
